# Emotion-Aware Multi-Artist Generative Art — Full Pipeline

**Project:** Emotion-Aware Multi-Artist Generative Art with AI Interpretation

**Artists:** Vincent van Gogh | Claude Monet | Leonardo da Vinci

**Pipeline stages covered in this notebook:**

1. Environment setup and dataset download
2. Exploratory Data Analysis (EDA) on the full dataset
3. Filter to the three target artists
4. Feature Engineering (pixel stats, metadata, derived features)
5. Emotion Labeling — Color-based heuristics
6. Emotion Labeling — CLIP-based similarity (optional, requires GPU)
7. GAN Preprocessing — resize to 256x256, normalize to [-1, 1], augmentation
8. Save GAN-ready dataset and labeled DataFrame

## Part 1 — Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn pillow scikit-learn kagglehub -q

import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

print("All libraries imported.")

## Part 2 — Download Dataset

In [ ]:
import kagglehub

path         = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")
csv_path     = os.path.join(path, "artists.csv")
resized_path = os.path.join(path, "resized", "resized")

print("Dataset path :", path)
print("CSV path     :", csv_path)
print("Images path  :", resized_path)

## Part 3 — EDA on Full Dataset

In [ ]:
artists_meta = pd.read_csv(csv_path)
print("CSV shape:", artists_meta.shape)
print("Columns  :", artists_meta.columns.tolist())
artists_meta.head()

In [ ]:
print("Missing values:")
print(artists_meta.isnull().sum())

In [ ]:
all_files = [f for f in os.listdir(resized_path) if f.endswith(".jpg")]
print("Total images in dataset:", len(all_files))

In [ ]:
# Count images per artist across the full dataset
artist_counts = {}
for fname in all_files:
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    name  = parts[0] if len(parts) == 2 and parts[1].isdigit() else fname.replace(".jpg", "")
    artist_counts[name] = artist_counts.get(name, 0) + 1

df_counts = pd.DataFrame(list(artist_counts.items()), columns=["Artist", "Image_Count"])
df_counts = df_counts.sort_values("Image_Count", ascending=False).reset_index(drop=True)

plt.figure(figsize=(14, 7))
sns.barplot(x="Image_Count", y="Artist", data=df_counts.head(20), palette="viridis")
plt.title("Top 20 Artists by Number of Images")
plt.xlabel("Number of Images")
plt.tight_layout()
plt.show()

print("Total artists:", len(df_counts))
df_counts.head(10)

In [ ]:
# Inspect a random 9-image sample from the full dataset
plt.figure(figsize=(12, 12))
sample_files = random.sample(all_files, 9)
for i, fname in enumerate(sample_files):
    img = Image.open(os.path.join(resized_path, fname))
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title("_".join(fname.split("_")[:-1]), fontsize=7)
    plt.axis("off")
plt.suptitle("Random Sample — Full Dataset", fontsize=14)
plt.tight_layout()
plt.show()

## Part 4 — Filter to Target Artists

We keep only the three artists relevant to this project:
- Vincent van Gogh
- Claude Monet
- Leonardo da Vinci

In [ ]:
TARGET_ARTISTS = ["Vincent_van_Gogh", "Claude_Monet", "Leonardo_da_Vinci"]

records = []
for fname in all_files:
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        artist_raw = parts[0]
        artwork_id = int(parts[1])
    else:
        continue
    if artist_raw not in TARGET_ARTISTS:
        continue
    records.append({
        "filename"   : fname,
        "filepath"   : os.path.join(resized_path, fname),
        "artist_raw" : artist_raw,
        "artist_name": artist_raw.replace("_", " "),
        "artwork_id" : artwork_id,
    })

df = pd.DataFrame(records)
print("Filtered dataset shape:", df.shape)
print(df["artist_name"].value_counts())

In [ ]:
# Show 3 sample images per artist
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Sample Images — Three Target Artists", fontsize=14, fontweight="bold")

for row_idx, artist in enumerate(TARGET_ARTISTS):
    subset = df[df["artist_raw"] == artist].sample(3, random_state=row_idx)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        img = Image.open(row["filepath"])
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(row["artist_name"], fontsize=8)
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Per-artist image count bar chart
plt.figure(figsize=(7, 4))
df["artist_name"].value_counts().plot(kind="bar", color=["steelblue", "seagreen", "tomato"])
plt.title("Image Count per Target Artist")
plt.ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Part 5 — Merge artists.csv Metadata

In [ ]:
artists_meta["artist_raw"] = artists_meta["name"].str.replace(" ", "_")

meta_slim = artists_meta[["artist_raw", "years", "genre", "nationality", "paintings"]].copy()
df = df.merge(meta_slim, on="artist_raw", how="left")

# Parse birth/death year
def parse_years(s):
    if pd.isna(s): return pd.Series([None, None, None])
    nums = re.findall(r"\d{4}", str(s))
    birth = int(nums[0]) if len(nums) >= 1 else None
    death = int(nums[1]) if len(nums) >= 2 else None
    age   = (death - birth) if birth and death else None
    return pd.Series([birth, death, age])

df[["birth_year", "death_year", "age_at_death"]] = df["years"].apply(parse_years)

print("Shape after metadata merge:", df.shape)
df[["artist_name", "genre", "nationality", "birth_year", "death_year"]]\
    .drop_duplicates("artist_name")

## Part 6 — Feature Engineering — Pixel-Level Statistics

We extract 14 numerical features from each image's raw pixels.

| Feature | Description |
|---------|-------------|
| img_width, img_height | Dimensions in pixels |
| aspect_ratio | width / height |
| mean_r, mean_g, mean_b | Average value per RGB channel |
| std_r, std_g, std_b | Standard deviation per RGB channel |
| brightness | Mean of grayscale-converted image |
| contrast | Std of grayscale-converted image |
| saturation | Mean of S channel in HSV |
| hue_mean | Mean of H channel in HSV |
| colorfulness | Hasler and Suesstrunk (2003) colorfulness metric |

In [ ]:
def extract_pixel_features(filepath):
    try:
        img_rgb = Image.open(filepath).convert("RGB")
        arr     = np.array(img_rgb, dtype=np.float32)
        h, w, _ = arr.shape
        r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
        gray    = 0.299*r + 0.587*g + 0.114*b
        hsv     = np.array(Image.open(filepath).convert("HSV"), dtype=np.float32)
        rg      = r - g
        yb      = 0.5*(r + g) - b
        cf      = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)
        return {
            "img_width"   : w,
            "img_height"  : h,
            "aspect_ratio": round(w/h, 4),
            "mean_r"      : round(r.mean(), 3),
            "mean_g"      : round(g.mean(), 3),
            "mean_b"      : round(b.mean(), 3),
            "std_r"       : round(r.std(), 3),
            "std_g"       : round(g.std(), 3),
            "std_b"       : round(b.std(), 3),
            "brightness"  : round(gray.mean(), 3),
            "contrast"    : round(gray.std(), 3),
            "saturation"  : round(hsv[:,:,1].mean(), 3),
            "hue_mean"    : round(hsv[:,:,0].mean(), 3),
            "colorfulness": round(cf, 3),
        }
    except Exception:
        return None


print("Extracting pixel features for all", len(df), "images...")
feat_rows = []
for _, row in df.iterrows():
    f = extract_pixel_features(row["filepath"])
    if f:
        f["filename"] = row["filename"]
        feat_rows.append(f)

feat_df = pd.DataFrame(feat_rows)
df = df.merge(feat_df, on="filename", how="inner")
print("Done. Shape:", df.shape)

In [ ]:
# Derived features
df["is_portrait"]     = (df["img_height"] > df["img_width"]).astype(int)
df["is_colorful"]     = (df["colorfulness"] > df["colorfulness"].median()).astype(int)
df["color_temp"]      = df.apply(lambda r: "Warm" if r["mean_r"] > r["mean_b"] else "Cool", axis=1)
df["dominant_channel"]= df.apply(
    lambda r: "Red" if r["mean_r"] >= r["mean_g"] and r["mean_r"] >= r["mean_b"]
    else ("Green" if r["mean_g"] >= r["mean_b"] else "Blue"), axis=1
)

print("Derived features added.")
df[["filename", "artist_name", "brightness", "colorfulness",
    "is_portrait", "color_temp", "dominant_channel"]].head(8)

In [ ]:
# Per-artist pixel stats comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Pixel Statistics by Artist", fontsize=14, fontweight="bold")

for ax, metric, color in zip(axes,
    ["brightness", "saturation", "colorfulness"],
    ["coral", "steelblue", "mediumseagreen"]):
    df.boxplot(column=metric, by="artist_name", ax=ax)
    ax.set_title(metric.capitalize())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Pixel Statistics by Artist", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — pixel features
corr_cols = ["brightness", "contrast", "saturation", "colorfulness",
             "mean_r", "mean_g", "mean_b", "aspect_ratio", "is_portrait", "is_colorful"]
corr_cols = [c for c in corr_cols if c in df.columns]

plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True, linewidths=0.4)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Part 7 — Emotion Labeling — Method 1: Color-Based Heuristics

We assign one of three emotion labels to each image using pixel statistics.

The logic is grounded in color psychology research:

| Emotion | Signal |
|---------|--------|
| Joy | High brightness AND high colorfulness AND warm color temperature |
| Sadness | Low brightness AND cool color temperature (blue dominant) |
| Calm | Everything in between — moderate brightness and saturation |

This is a heuristic approach. It is fast, requires no GPU, and gives reasonable labels for training.

In [ ]:
def color_based_emotion(row):
    """
    Assign an emotion label using brightness, colorfulness, and color temperature.
    Thresholds are based on the dataset distribution (percentile-driven).
    """
    b  = row["brightness"]
    cf = row["colorfulness"]
    ct = row["color_temp"]

    if b >= 160 and cf >= 40 and ct == "Warm":
        return "Joy"
    elif b <= 100 and ct == "Cool":
        return "Sadness"
    else:
        return "Calm"

df["emotion_color"] = df.apply(color_based_emotion, axis=1)

print("Emotion distribution (color-based):")
print(df["emotion_color"].value_counts())

In [ ]:
# Emotion distribution per artist
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Emotion Distribution by Artist (Color-Based)", fontsize=13, fontweight="bold")

palette = {"Joy": "gold", "Calm": "steelblue", "Sadness": "mediumpurple"}

for ax, artist in zip(axes, [a.replace("_", " ") for a in TARGET_ARTISTS]):
    subset = df[df["artist_name"] == artist]
    counts = subset["emotion_color"].value_counts()
    counts.plot(kind="bar", ax=ax,
                color=[palette.get(e, "gray") for e in counts.index])
    ax.set_title(artist, fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize 3 images per emotion
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Sample Images by Emotion (Color-Based)", fontsize=14, fontweight="bold")

for row_idx, emotion in enumerate(["Joy", "Calm", "Sadness"]):
    subset = df[df["emotion_color"] == emotion].sample(3, random_state=42)
    for col_idx, (_, img_row) in enumerate(subset.iterrows()):
        img = Image.open(img_row["filepath"])
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(
            f"{emotion} | {img_row['artist_name'].split(' ')[-1]}", fontsize=8
        )
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
plt.show()

## Part 8 — Emotion Labeling — Method 2: CLIP-Based Similarity (Optional)

CLIP (Contrastive Language-Image Pretraining) is a model from OpenAI that understands
both images and text. We feed it each image and three text prompts:

- 'a joyful, bright, colorful painting'
- 'a calm, peaceful, serene painting'
- 'a sad, dark, melancholic painting'

CLIP returns a similarity score for each prompt. We assign the label with the highest score.

This method is more semantically accurate than color heuristics but requires a GPU and more time.

**How to enable:** Set RUN_CLIP = True below. Make sure you are using a Colab GPU runtime.

In [ ]:
RUN_CLIP = False  # Set to True to run CLIP-based labeling

if RUN_CLIP:
    !pip install open_clip_torch -q
    import torch
    import open_clip

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer("ViT-B-32")

    EMOTION_PROMPTS = [
        "a joyful, bright, colorful painting",
        "a calm, peaceful, serene painting",
        "a sad, dark, melancholic painting",
    ]
    EMOTION_NAMES = ["Joy", "Calm", "Sadness"]

    text_tokens = tokenizer(EMOTION_PROMPTS).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    def clip_emotion(filepath):
        try:
            img_tensor = preprocess(Image.open(filepath)).unsqueeze(0).to(device)
            with torch.no_grad():
                img_features = model.encode_image(img_tensor)
                img_features = img_features / img_features.norm(dim=-1, keepdim=True)
                scores = (img_features @ text_features.T).squeeze(0)
            return EMOTION_NAMES[scores.argmax().item()]
        except Exception:
            return None

    print("Running CLIP emotion labeling on", len(df), "images...")
    df["emotion_clip"] = df["filepath"].apply(clip_emotion)
    print("CLIP emotion distribution:")
    print(df["emotion_clip"].value_counts())

else:
    df["emotion_clip"] = None
    print("CLIP labeling skipped. Set RUN_CLIP = True to enable it.")

In [ ]:
# Choose which emotion label to use for training
# If CLIP was run, use emotion_clip. Otherwise fall back to color-based.

df["emotion_final"] = df["emotion_clip"].fillna(df["emotion_color"])

print("Final emotion label distribution:")
print(df["emotion_final"].value_counts())

print("\nMethod used per image:")
clip_count  = df["emotion_clip"].notna().sum()
color_count = df["emotion_clip"].isna().sum()
print(f"  CLIP-based  : {clip_count} images")
print(f"  Color-based : {color_count} images")

## Part 9 — GAN Preprocessing

Before training a GAN, images must be:

1. **Resized to 256x256** — fixed input size required by the GAN architecture
2. **Converted to RGB** — ensures no grayscale or RGBA images slip through
3. **Normalized to [-1, 1]** — GANs use tanh activation in the generator output layer,
   which produces values in [-1, 1]. Matching this range during training stabilizes learning.

Formula: `pixel_normalized = (pixel / 127.5) - 1.0`

We also apply optional data augmentation to reduce overfitting:
- Horizontal flip (50% probability)
- Small random rotation (+/- 15 degrees)
- Slight color jitter

In [ ]:
import torchvision.transforms as T
from torchvision.utils import save_image
import torch

GAN_IMAGE_SIZE = 256

# Standard GAN preprocessing transform
gan_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.ToTensor(),                             # [0, 255] -> [0.0, 1.0]
    T.Normalize(mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]),          # [0.0, 1.0] -> [-1.0, 1.0]
])

# Augmentation transform (used only during GAN training, not saved to disk)
aug_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("GAN transforms defined.")
print("Target image size:", GAN_IMAGE_SIZE, "x", GAN_IMAGE_SIZE)
print("Normalization range: [-1, 1]")

In [ ]:
# Verify preprocessing on a single image
sample_path = df.iloc[0]["filepath"]
raw_img     = Image.open(sample_path).convert("RGB")
tensor_img  = gan_transform(raw_img)

print("Original size :", raw_img.size)
print("Tensor shape  :", tensor_img.shape)   # Should be [3, 256, 256]
print("Tensor min    :", round(tensor_img.min().item(), 4))   # Should be close to -1
print("Tensor max    :", round(tensor_img.max().item(), 4))   # Should be close to +1

# Visualize before / after
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(raw_img)
axes[0].set_title("Original")
axes[0].axis("off")

# Denormalize for display: pixel = (tensor + 1) / 2
display_tensor = (tensor_img + 1) / 2
display_img    = T.ToPILImage()(display_tensor)
axes[1].imshow(display_img)
axes[1].set_title("After GAN Preprocessing (256x256, RGB)")
axes[1].axis("off")

plt.suptitle("GAN Preprocessing Verification", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Save preprocessed images to disk, organized by artist and emotion
OUTPUT_DIR = "/content/gan_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved = 0
skipped = 0

for _, row in df.iterrows():
    artist_dir  = os.path.join(OUTPUT_DIR, row["artist_raw"], row["emotion_final"])
    os.makedirs(artist_dir, exist_ok=True)

    out_path = os.path.join(artist_dir, row["filename"])
    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        img    = Image.open(row["filepath"]).convert("RGB")
        tensor = gan_transform(img)
        # Denormalize back to [0, 1] for saving as PNG
        save_image((tensor + 1) / 2, out_path)
        saved += 1
    except Exception:
        skipped += 1

print(f"Preprocessing complete.")
print(f"Saved : {saved} images")
print(f"Skipped: {skipped} images")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Structure: gan_dataset / artist / emotion / image.jpg")

In [ ]:
# Verify the folder structure
for artist in os.listdir(OUTPUT_DIR):
    artist_path = os.path.join(OUTPUT_DIR, artist)
    if os.path.isdir(artist_path):
        for emotion in os.listdir(artist_path):
            emotion_path = os.path.join(artist_path, emotion)
            count = len(os.listdir(emotion_path))
            print(f"{artist:30s} | {emotion:8s} | {count} images")

## Part 10 — Encode, Normalize, and Save Final DataFrame

In [ ]:
# Label encode artist name and emotion
le = LabelEncoder()
df["artist_enc"]  = le.fit_transform(df["artist_name"])
df["emotion_enc"] = le.fit_transform(df["emotion_final"])

# One-hot encode emotion for direct use in conditioning
df = pd.get_dummies(df, columns=["emotion_final"], prefix="emotion")

# Normalize numerical pixel features to [0, 1]
num_cols = ["brightness", "contrast", "saturation", "colorfulness",
            "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
            "aspect_ratio", "hue_mean"]
num_cols = [c for c in num_cols if c in df.columns]

scaler = MinMaxScaler()
df[[f"{c}_norm" for c in num_cols]] = scaler.fit_transform(df[num_cols])

# Drop helper columns not needed downstream
drop_cols = [c for c in ["filepath", "artist_raw", "years", "emotion_color", "emotion_clip"]
             if c in df.columns]
df_final = df.drop(columns=drop_cols)

# Save
csv_out = "/content/artworks_gan_ready.csv"
df_final.to_csv(csv_out, index=False)

print("Saved:", csv_out)
print("Shape:", df_final.shape)
print("\nAll columns:")
print(df_final.columns.tolist())

## Pipeline Summary

| Stage | What was done |
|-------|---------------|
| EDA | Explored full dataset of 8683 images across 50 artists |
| Filtering | Kept only Van Gogh, Monet, and Leonardo da Vinci |
| Metadata Merge | Added genre, nationality, birth/death year from artists.csv |
| Pixel Features | Extracted 14 statistics per image: brightness, contrast, saturation, RGB stats, colorfulness |
| Derived Features | is_portrait, is_colorful, color_temp, dominant_channel |
| Emotion Labeling (Color) | Joy / Calm / Sadness assigned via brightness, colorfulness, and color temperature |
| Emotion Labeling (CLIP) | Optional: CLIP similarity to text prompts per emotion (requires GPU) |
| GAN Preprocessing | Resized to 256x256, normalized to [-1, 1], augmentation transforms defined |
| Saved to Disk | Images organized as gan_dataset/artist/emotion/image.jpg |
| Final DataFrame | All features encoded and normalized, saved to artworks_gan_ready.csv |

**Next step:** Load `gan_dataset/` into a PyTorch Dataset and begin GAN training.